# Diabetes Prediction – Capstone 1

This notebook covers:
1. Data loading
2. EDA
3. Preprocessing
4. Model training & comparison
5. Final model selection notes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

df = pd.read_csv('data/diabetes.csv')
df.head()


## Basic checks


In [ ]:
df.shape, df.columns


In [ ]:
df.info()


## Target distribution


In [ ]:
target_col = 'Outcome'
df[target_col].value_counts(), df[target_col].value_counts(normalize=True)


In [ ]:
counts = df[target_col].value_counts().sort_index()
plt.figure(figsize=(5,4))
plt.bar(counts.index.astype(str), counts.values)
plt.title('Target distribution (Outcome)')
plt.xlabel('Outcome (0=no, 1=yes)')
plt.ylabel('Count')
plt.show()


## Zero-as-missing analysis


In [ ]:
zero_as_missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
summary = pd.DataFrame({
    'zero_count': (df[zero_as_missing_cols] == 0).sum(),
    'zero_percent': (df[zero_as_missing_cols] == 0).mean() * 100
}).sort_values('zero_percent', ascending=False)
summary


## Correlations


In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(9,7))
plt.imshow(corr.values, aspect='auto')
plt.title('Correlation heatmap')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.tight_layout()
plt.show()

corr['Outcome'].sort_values(ascending=False)


## Preprocessing + model comparison (CV ROC-AUC)


In [ ]:
df2 = df.copy()
for c in zero_as_missing_cols:
    df2.loc[df2[c] == 0, c] = np.nan

y = df2['Outcome'].astype(int)
X = df2.drop(columns=['Outcome'])
numeric_cols = list(X.columns)

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer([('num', num_pipe, numeric_cols)])

models = {
    'logreg': LogisticRegression(max_iter=500),
    'rf': RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    'gb': GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for name, m in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', m)])
    auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc').mean()
    scores.append((name, auc))

pd.DataFrame(scores, columns=['model', 'roc_auc']).sort_values('roc_auc', ascending=False)
